# Control de calidad de manzanas: segmentación por instancia y estado (sana / deteriorada)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EMatCM/proyectoAprendizaje/blob/main/notebooks/baseline.ipynb)

**Qué hace este notebook**

Segmenta cada manzana de una foto con varias, clasifica cada una como sana o deteriorada con el **modelo final** (`modelos/finalbaseline.pt`), cuenta el total, las sanas y las deterioradas, dibuja el resultado, evalúa con tus fotos anotadas y puede abrir una interfaz Gradio.

Aquí **no se entrena nada** y **no se descarga el dataset**. El entrenamiento y los experimentos están en `notebooks/experimentos.ipynb`.

**Componentes externos:** ResNet18 con pesos de ImageNet (la arquitectura del clasificador) y Mask R-CNN con pesos de COCO (el detector, que no se entrena), ambos vía torchvision. Lo propio: el clasificador de estado entrenado, el filtrado de detecciones, las métricas y el análisis.

## 1. Clonar el repositorio (código, modelo y resultados)

In [ ]:
import os
REPO = "/content/proyectoAprendizaje"
if not os.path.exists(REPO):
    !git clone https://github.com/EMatCM/proyectoAprendizaje.git {REPO}
%cd {REPO}
!git pull
!ls modelos

## 2. Importaciones

In [ ]:
# Colab ya trae torch, torchvision y pillow. Fuera de Colab: pip install -r requirements.txt
import json, random, time
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

## 3. Interruptores, parámetros y rutas

In [ ]:
# ---- Interruptor ----
LANZAR_GRADIO = False       # True para abrir la interfaz Gradio

# ---- Parámetros ----
SEMILLA = 42
TAM = 224
MEDIA, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CLASE_MALA = "rottenapples"   # clase positiva (la que se cuenta como deteriorada)

# ---- Rutas ----
REPO = "/content/proyectoAprendizaje"
MODELO_FINAL = Path(f"{REPO}/modelos/finalbaseline.pt")   # clasificador de estado (modelo final)

def fijar_semilla(s=SEMILLA):
    random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
fijar_semilla()
print("Dispositivo:", dev)

## 4. Transformaciones y modelo

In [ ]:
def tf_eval():
    return transforms.Compose([
        transforms.Resize((TAM, TAM)),
        transforms.ToTensor(),
        transforms.Normalize(MEDIA, STD),
    ])

def crear_modelo(n_clases=2, congelar=True):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if congelar:
        for p in m.parameters():
            p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, n_clases)
    return m

# Varias manzanas por foto

Flujo: **Mask R-CNN** (COCO) encuentra cada manzana y su máscara → se recorta cada una con fondo blanco → el **clasificador final** decide sana o deteriorada → se cuenta el total, las sanas y las deterioradas.

Límites conocidos: las manzanas muy podridas pueden no ser detectadas como `apple`, las que se tocan o se tapan pueden fusionarse o quedar con la máscara cortada, y en montones muy llenos el detector no encuentra todas. Revisa a mano los resultados con confianza baja.

### Parámetros de la segmentación

In [ ]:
import os, time
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.transforms.functional import to_tensor

# ---- Parámetros de la segmentación ----
UMBRAL_DET = 0.5        # confianza para aceptar una manzana sin más condiciones
UMBRAL_RESCATE = 0.35   # entre este valor y UMBRAL_DET solo se acepta si NO se pega a otra manzana (rescata podridas dudosas)
UMBRAL_MASCARA = 0.5    # umbral para volver binaria la máscara suave del detector
UMBRAL_MALA = 0.5       # probabilidad mínima para decir "deteriorada"
CONF_BAJA = 0.6         # por debajo de esta confianza se avisa que el resultado es incierto
MARGEN = 0.08           # margen extra alrededor de cada recorte (8 %)
MAX_LADO = 1024         # las fotos más grandes se reducen a este lado máximo
MIN_AREA_FRAC = 0.002   # descarta detecciones menores al 0,2 % de la imagen
FRAC_CONTENIDA = 0.8    # una máscara 'contiene' a otra si cubre al menos este % del área de la otra
MAX_CONTENIDAS = 2      # una detección que contiene a 2 o más manzanas es un grupo, no una manzana
MAX_AREA_REL = 4.0      # con 6 o más manzanas seguras, una detección más grande que 4 veces la mediana es un grupo (None = desactivar)
IOU_DUPLICADA = 0.6     # dos máscaras con IoU >= esto son la misma manzana: se queda la de mayor puntaje
SOLAPE_RESCATE = 0.3    # una detección dudosa se descarta si más de este % de su área cae sobre otra ya aceptada
COLOR = {"sana": (40, 170, 60), "deteriorada": (220, 40, 40)}

### Cargar clasificador y detector

In [ ]:
# ---- Clasificador de estado (modelo final) ----
assert MODELO_FINAL.exists(), (
    f"No existe {MODELO_FINAL}. Modelos disponibles: "
    f"{sorted(p.name for p in MODELO_FINAL.parent.glob('*.pt'))}. "
    "Sube el modelo a modelos/ en el repositorio y vuelve a ejecutar la celda de clonado.")
ck_clf = torch.load(MODELO_FINAL, map_location=dev)
clases_clf = ck_clf.get("clases", ["freshapples", "rottenapples"])
assert CLASE_MALA in clases_clf, f"Clases del modelo: {clases_clf}"
clasificador = crear_modelo(len(clases_clf), congelar=False)
clasificador.load_state_dict(ck_clf["state_dict"])
clasificador = clasificador.to(dev).eval()
print("Clasificador:", MODELO_FINAL.name, "| Clases:", clases_clf)

# ---- Detector / segmentador (externo: Mask R-CNN preentrenado en COCO, no se entrena) ----
pesos_det = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
detector = maskrcnn_resnet50_fpn_v2(weights=pesos_det).to(dev).eval()
ID_MANZANA = pesos_det.meta["categories"].index("apple")
print("Detector cargado | id de 'apple' en COCO:", ID_MANZANA)

# El detector COCO llama 'donut', 'orange' o 'cake' a muchas manzanas muy podridas (marrones y arrugadas).
# Se aceptan esas clases como "posible manzana"; 'bowl' y otras no, porque darían falsos positivos.
CLASES_FRUTA = ["apple", "orange", "donut", "cake"]
IDS_FRUTA = torch.tensor([pesos_det.meta["categories"].index(c) for c in CLASES_FRUTA], device=dev)
print("Clases aceptadas como manzana:", CLASES_FRUTA)

### Segmentar, recortar y clasificar

In [ ]:
def filtrar_instancias(puntajes, mascaras, umbral_det):
    """Índices (por puntaje decreciente) de las detecciones que son UNA manzana.

    Trabaja con las máscaras, no con las cajas: una caja grande puede tocar muchas manzanas sin ser un grupo.
    1) quita grupos: máscaras que contienen a 2 o más manzanas seguras, o que son enormes frente a las demás;
    2) quita duplicados: de dos máscaras casi iguales deja la de mayor puntaje;
    3) las de puntaje dudoso (entre UMBRAL_RESCATE y umbral_det) solo se aceptan si no se montan sobre otra.
    """
    n = len(puntajes)
    if n == 0:
        return np.zeros(0, dtype=int)
    M = np.asarray(mascaras).reshape(n, -1).astype(np.float32)
    area = M.sum(1)
    inter = M @ M.T                                   # píxeles en común entre cada par de máscaras
    seguras = np.asarray(puntajes) >= umbral_det
    vivos = np.ones(n, dtype=bool)
    mediana = np.median(area[seguras]) if seguras.sum() >= 6 else None
    for i in range(n):
        dentro = sum(1 for j in range(n)
                     if j != i and seguras[j] and area[j] < 0.6 * area[i] and inter[i, j] >= FRAC_CONTENIDA * area[j])
        enorme = MAX_AREA_REL is not None and mediana is not None and area[i] > MAX_AREA_REL * mediana
        if dentro >= MAX_CONTENIDAS or enorme:
            vivos[i] = False
    kept = []
    for i in np.argsort(-np.asarray(puntajes), kind="stable"):
        if not vivos[i]:
            continue
        if kept:
            k = np.array(kept)
            iou = inter[i, k] / (area[i] + area[k] - inter[i, k])
            if (iou >= IOU_DUPLICADA).any():
                continue
            if puntajes[i] < umbral_det and (inter[i, k] / area[i] > SOLAPE_RESCATE).any():
                continue
        kept.append(int(i))
    return np.array(kept, dtype=int)


@torch.no_grad()
def segmentar(img, umbral_det=None):
    """Devuelve cajas, puntajes y máscaras booleanas de cada manzana detectada."""
    umbral_det = UMBRAL_DET if umbral_det is None else umbral_det
    piso = min(UMBRAL_RESCATE, umbral_det)            # el detector entrega también las dudosas; luego se filtran
    out = detector([to_tensor(img).to(dev)])[0]
    ok = torch.isin(out["labels"], IDS_FRUTA) & (out["scores"] >= piso)
    cajas = out["boxes"][ok].cpu().numpy()
    puntajes = out["scores"][ok].cpu().numpy()
    mascaras = (out["masks"][ok, 0] > UMBRAL_MASCARA).cpu().numpy()
    area_min = MIN_AREA_FRAC * img.width * img.height
    keep = np.array([i for i, m in enumerate(mascaras) if m.sum() >= area_min], dtype=int)
    cajas, puntajes, mascaras = cajas[keep], puntajes[keep], mascaras[keep]
    keep = filtrar_instancias(puntajes, mascaras, umbral_det)
    return cajas[keep], puntajes[keep], mascaras[keep]


def recortar(arr, caja, mascara, margen=MARGEN):
    """Recorta una manzana y pone fondo blanco fuera de su máscara."""
    x1, y1, x2, y2 = caja
    mx, my = margen * (x2 - x1), margen * (y2 - y1)
    x1, y1 = max(0, int(x1 - mx)), max(0, int(y1 - my))
    x2, y2 = min(arr.shape[1], int(x2 + mx)), min(arr.shape[0], int(y2 + my))
    sub = arr[y1:y2, x1:x2].copy()
    sub[~mascara[y1:y2, x1:x2]] = 255
    return Image.fromarray(sub)


@torch.no_grad()
def prob_deteriorada(recortes, lote=32):
    """Probabilidad de 'deteriorada' para cada recorte (el mismo clasificador del baseline)."""
    if not recortes:
        return []
    idx = clases_clf.index(CLASE_MALA)
    probs = []
    for i in range(0, len(recortes), lote):
        x = torch.stack([tf_eval()(r) for r in recortes[i:i + lote]]).to(dev)
        probs += torch.softmax(clasificador(x), dim=1)[:, idx].cpu().tolist()
    return probs

### Análisis de una imagen y resumen

In [ ]:
def analizar(img, umbral_mala=None, umbral_det=None):
    """Segmenta, clasifica cada manzana y arma el resumen de la imagen."""
    umbral_mala = UMBRAL_MALA if umbral_mala is None else umbral_mala
    t0 = time.time()
    img = img.convert("RGB")
    factor = 1.0
    if max(img.size) > MAX_LADO:                      # fotos de celular muy grandes
        factor = MAX_LADO / max(img.size)
        img = img.resize((round(img.width * factor), round(img.height * factor)))

    cajas, puntajes, mascaras = segmentar(img, umbral_det)
    arr = np.asarray(img)
    recortes = [recortar(arr, c, m) for c, m in zip(cajas, mascaras)]
    probs = prob_deteriorada(recortes)

    instancias = []
    for i, (c, s, m, p) in enumerate(zip(cajas, puntajes, mascaras, probs), 1):
        mala = p >= umbral_mala
        instancias.append({
            "id": i,
            "caja": [float(v) for v in c],
            "puntaje_det": float(s),
            "prob_deteriorada": float(p),
            "estado": "deteriorada" if mala else "sana",
            "confianza": float(p if mala else 1 - p),
            "area": int(m.sum()),
            "mascara": m,
        })

    n_malas = sum(d["estado"] == "deteriorada" for d in instancias)
    avisos = []
    if not instancias:
        avisos.append("No se detectó ninguna manzana. Prueba bajar el umbral de detección.")
    dudosas = [d["id"] for d in instancias if d["confianza"] < CONF_BAJA]
    if dudosas:
        avisos.append(f"Confianza baja en las manzanas {dudosas}: revisar a mano.")
    return {"imagen": img, "factor": factor, "instancias": instancias,
            "total": len(instancias), "sanas": len(instancias) - n_malas,
            "deterioradas": n_malas, "avisos": avisos, "segundos": time.time() - t0}


def resumen_texto(res):
    lineas = [f"Manzanas: {res['total']} | sanas: {res['sanas']} | deterioradas: {res['deterioradas']} "
              f"| {res['segundos']:.1f} s"]
    for d in res["instancias"]:
        lineas.append(f"  #{d['id']}: {d['estado']} (confianza {d['confianza']:.0%}, "
                      f"detección {d['puntaje_det']:.0%})")
    lineas += [f"Aviso: {a}" for a in res["avisos"]]
    return "\n".join(lineas)

### Dibujar el resultado

In [ ]:
def _borde(m, grosor=3):
    """Contorno de una máscara booleana, sin necesidad de OpenCV."""
    interior = m.copy()
    for _ in range(grosor):
        p = np.pad(interior, 1, constant_values=False)
        interior = p[1:-1, 1:-1] & p[:-2, 1:-1] & p[2:, 1:-1] & p[1:-1, :-2] & p[1:-1, 2:]
    return m & ~interior


def _fuente(tam):
    for ruta in ["/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", "DejaVuSans-Bold.ttf"]:
        try:
            return ImageFont.truetype(ruta, tam)
        except OSError:
            pass
    return ImageFont.load_default()


def dibujar(res, alpha=0.40):
    """Pinta cada manzana (verde = sana, rojo = deteriorada) con contorno y etiqueta."""
    arr = np.asarray(res["imagen"]).astype(np.float32)
    for d in res["instancias"]:
        color = COLOR[d["estado"]]
        m = d["mascara"]
        arr[m] = (1 - alpha) * arr[m] + alpha * np.array(color, dtype=np.float32)
        arr[_borde(m)] = color
    out = Image.fromarray(arr.clip(0, 255).astype(np.uint8))
    dr = ImageDraw.Draw(out)
    fuente = _fuente(max(14, out.width // 45))
    for d in res["instancias"]:
        color = COLOR[d["estado"]]
        texto = f"#{d['id']} {d['estado']} {d['confianza']:.0%}"
        x1, y1 = d["caja"][:2]
        bb = dr.textbbox((0, 0), texto, font=fuente)
        w, h = bb[2] - bb[0], bb[3] - bb[1]
        px, py = x1, max(0, y1 - h - 10)
        dr.rectangle([px, py, px + w + 10, py + h + 10], fill=color)
        dr.text((px + 5, py + 3), texto, fill="white", font=fuente)
    return out

### Probar con una foto de varias manzanas

In [ ]:
from google.colab import files
from IPython.display import display

subida = files.upload()          # sube una foto con varias manzanas
if not subida:
    print("No se subió ninguna imagen. Vuelve a ejecutar esta celda y elige un archivo.")
else:
    ruta = list(subida.keys())[0]
    res = analizar(Image.open(ruta))
    print(resumen_texto(res))
    display(dibujar(res))
    os.remove(ruta)              # para que la foto no quede dentro del repositorio

### Evaluación por manzana (con tus fotos anotadas)

Necesita fotos propias con **máscara y estado reales por manzana**, exportadas en **formato COCO con polígonos** (Roboflow, CVAT o X-AnyLabeling). Ajusta `ESTADO_POR_CATEGORIA` a los nombres de tus clases.

In [ ]:
import json
from pathlib import Path

# Nombre de la categoría en tu exportación COCO -> estado. Ajústalo a como llamaste tus clases.
ESTADO_POR_CATEGORIA = {"sana": "sana", "deteriorada": "deteriorada"}


def cargar_coco(ruta_json):
    """Lee anotaciones COCO con polígonos. Devuelve {archivo: [{poligonos, estado}]}."""
    coco = json.load(open(ruta_json, encoding="utf-8"))
    cats = {c["id"]: c["name"] for c in coco["categories"]}
    nombres = {i["id"]: i["file_name"] for i in coco["images"]}
    gt = {i["file_name"]: [] for i in coco["images"]}
    omitidas = 0
    for a in coco["annotations"]:
        seg = a.get("segmentation")
        cat = cats.get(a["category_id"])
        if not isinstance(seg, list) or not seg or cat not in ESTADO_POR_CATEGORIA:
            omitidas += 1               # RLE, vacías o de otra categoría
            continue
        gt[nombres[a["image_id"]]].append({"poligonos": seg, "estado": ESTADO_POR_CATEGORIA[cat]})
    if omitidas:
        print(f"Aviso: {omitidas} anotaciones omitidas (sin polígono o categoría desconocida).")
    return gt


def rasterizar(poligonos, ancho, alto, factor):
    """Polígonos COCO -> máscara booleana, a la misma escala que la imagen analizada."""
    im = Image.new("L", (ancho, alto), 0)
    d = ImageDraw.Draw(im)
    for poly in poligonos:
        pts = [(poly[i] * factor, poly[i + 1] * factor) for i in range(0, len(poly) - 1, 2)]
        if len(pts) >= 3:
            d.polygon(pts, fill=1)
    return np.array(im, dtype=bool)


def iou(a, b):
    union = (a | b).sum()
    return float((a & b).sum() / union) if union else 0.0


def emparejar(preds, gts, iou_min=0.5):
    """Empareja predicciones con anotaciones reales, de mayor a menor IoU (uno a uno)."""
    cand = sorted(((iou(p["mascara"], g["mascara"]), i, j)
                   for i, p in enumerate(preds) for j, g in enumerate(gts)), reverse=True)
    pares, usados_p, usados_g = [], set(), set()
    for v, i, j in cand:
        if v < iou_min:
            break
        if i in usados_p or j in usados_g:
            continue
        pares.append((i, j, v))
        usados_p.add(i)
        usados_g.add(j)
    return pares, usados_p, usados_g


def _div(a, b):
    return a / b if b else 0.0


def evaluar_coco(ruta_json, carpeta_imgs, iou_min=0.5, umbral_mala=None, guardar=None):
    """Evalúa por manzana: detección, IoU, conteo y estado, contra tus anotaciones reales."""
    gt = cargar_coco(ruta_json)
    det_tp = det_fp = det_fn = 0
    ious, err_total, err_malas, filas = [], [], [], []
    conf = {"TP": 0, "FP": 0, "FN": 0, "TN": 0}      # estado, solo en manzanas emparejadas
    malas_reales = malas_ok_extremo = 0
    for archivo, anots in gt.items():
        res = analizar(Image.open(Path(carpeta_imgs) / Path(archivo).name), umbral_mala)
        w, h = res["imagen"].size
        gts = [{"mascara": rasterizar(a["poligonos"], w, h, res["factor"]), "estado": a["estado"]}
               for a in anots]
        pares, up, ug = emparejar(res["instancias"], gts, iou_min)
        det_tp += len(pares)
        det_fp += len(res["instancias"]) - len(up)
        det_fn += len(gts) - len(ug)
        errores_estado = 0
        for i, j, v in pares:
            ious.append(v)
            pred, real = res["instancias"][i]["estado"], gts[j]["estado"]
            if real == "deteriorada":
                malas_reales += 1
            if pred == "deteriorada" and real == "deteriorada":
                conf["TP"] += 1; malas_ok_extremo += 1
            elif pred == "deteriorada":
                conf["FP"] += 1; errores_estado += 1
            elif real == "deteriorada":
                conf["FN"] += 1; errores_estado += 1
            else:
                conf["TN"] += 1
        malas_reales += sum(1 for j, g in enumerate(gts) if j not in ug and g["estado"] == "deteriorada")
        real_malas = sum(g["estado"] == "deteriorada" for g in gts)
        err_total.append(abs(res["total"] - len(gts)))
        err_malas.append(abs(res["deterioradas"] - real_malas))
        filas.append({"imagen": archivo, "reales": len(gts), "predichas": res["total"],
                      "no_detectadas": len(gts) - len(ug), "de_mas": len(res["instancias"]) - len(up),
                      "errores_de_estado": errores_estado,
                      "malas_reales": real_malas, "malas_predichas": res["deterioradas"]})

    prec_e, rec_e = _div(conf["TP"], conf["TP"] + conf["FP"]), _div(conf["TP"], conf["TP"] + conf["FN"])
    metricas = {
        "n_imagenes": len(gt), "n_manzanas_reales": det_tp + det_fn, "iou_minimo": iou_min,
        "deteccion": {"TP": det_tp, "FP": det_fp, "FN": det_fn,
                      "precision": _div(det_tp, det_tp + det_fp), "recall": _div(det_tp, det_tp + det_fn),
                      "iou_medio": _div(sum(ious), len(ious))},
        "conteo": {"error_abs_medio_total": _div(sum(err_total), len(err_total)),
                   "error_abs_medio_deterioradas": _div(sum(err_malas), len(err_malas)),
                   "imagenes_con_conteo_exacto": sum(e == 0 for e in err_total)},
        "estado_en_emparejadas": {**conf, "precision_deteriorada": prec_e, "recall_deteriorada": rec_e,
                                  "f1_deteriorada": _div(2 * prec_e * rec_e, prec_e + rec_e)},
        "recall_deteriorada_extremo_a_extremo": _div(malas_ok_extremo, malas_reales),
        "por_imagen": filas,
    }
    if guardar:
        Path(guardar).parent.mkdir(parents=True, exist_ok=True)
        Path(guardar).write_text(json.dumps(metricas, indent=2, ensure_ascii=False))
        print("Resultados guardados en", guardar)
    return metricas

In [ ]:
# Ejecuta la evaluación cuando ya tengas tus fotos y el JSON COCO en estas rutas
RUTA_FOTOS_PROPIAS = "/content/mis_fotos"
RUTA_COCO = f"{RUTA_FOTOS_PROPIAS}/_annotations.coco.json"

if os.path.exists(RUTA_COCO):
    met = evaluar_coco(RUTA_COCO, RUTA_FOTOS_PROPIAS,
                       guardar=f"{REPO}/resultados/evaluacion_segmentacion.json")
    print(json.dumps({k: v for k, v in met.items() if k != "por_imagen"}, indent=2, ensure_ascii=False))
else:
    print("Aún no hay fotos anotadas en", RUTA_FOTOS_PROPIAS, "- evaluación omitida.")


### Interfaz Gradio

In [ ]:
if LANZAR_GRADIO:
    !pip install -q gradio
    import gradio as gr

    def interfaz(img, umbral_mala, umbral_det):
        if img is None:
            return None, "Sube una foto."
        res = analizar(img, umbral_mala, umbral_det)
        return dibujar(res), resumen_texto(res)

    demo = gr.Interface(
        fn=interfaz,
        inputs=[gr.Image(type="pil", label="Foto con varias manzanas"),
                gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="Umbral para 'deteriorada'"),
                gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="Umbral de detección")],
        outputs=[gr.Image(label="Resultado"), gr.Textbox(label="Resumen", lines=8)],
        title="Control de calidad de manzanas",
        description="Segmenta cada manzana y la clasifica como sana o deteriorada. Inspección solo visual.",
    )
    demo.launch(share=True)
else:
    print("Interfaz omitida (LANZAR_GRADIO = False).")

## Dónde está el resto

- Entrenamiento del clasificador, experimentos y comparaciones: `notebooks/experimentos.ipynb`.
- Modelos y métricas guardadas: carpetas `modelos/` y `resultados/` del repositorio.